In [1]:
import hashlib, json, os, re, sqlite3, time, csv
from datetime import datetime, timezone
from pathlib import Path
import requests

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    pass  # خارج Colab: يؤخذ من متغيرات البيئة / .env

assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY manquante"

In [2]:
MODEL = "openai/gpt-oss-120b"
URL = "https://api.groq.com/openai/v1/chat/completions"
DB = Path("data/processed/llm_cache.sqlite")
META = Path("data/processed/experiment_metadata.json")
CALLS = {"real": 0, "cached": 0}
DEFAULT_CONFIG = {"temperature": 0, "seed": 42}

def _key(prompt, config):
    return hashlib.sha256((MODEL + json.dumps(config, sort_keys=True) + prompt).encode()).hexdigest()

def generate(prompt, config=None):
    cfg = {**DEFAULT_CONFIG, **(config or {})}
    DB.parent.mkdir(parents=True, exist_ok=True)
    con = sqlite3.connect(DB)
    con.execute("""CREATE TABLE IF NOT EXISTS cache (
        k TEXT PRIMARY KEY, prompt TEXT, model TEXT, model_id TEXT,
        config TEXT, generated_at TEXT, response TEXT)""")
    k = _key(prompt, cfg)
    row = con.execute("SELECT response FROM cache WHERE k=?", (k,)).fetchone()
    if row:
        CALLS["cached"] += 1
        con.close()
        return row[0]

    body = {"model": MODEL, "messages": [{"role": "user", "content": prompt}], **cfg}
    headers = {"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
    for _ in range(5):
        r = requests.post(URL, headers=headers, json=body, timeout=120)
        if r.status_code == 429:
            time.sleep(int(float(r.headers.get("Retry-After", 5))) + 1)
            continue
        r.raise_for_status()
        break
    else:
        raise RuntimeError("Échec : limite de requêtes (429) persistante")

    data = r.json()
    text = data["choices"][0]["message"]["content"]
    CALLS["real"] += 1
    con.execute("INSERT INTO cache VALUES (?,?,?,?,?,?,?)",
                (k, prompt, MODEL, data.get("model"), json.dumps(cfg),
                 datetime.now(timezone.utc).isoformat(), text))
    con.commit(); con.close()
    return text

In [3]:
SITUATION = """Ahmed travaille dans une entreprise privée en Algérie depuis trois ans
dans le cadre d’un contrat à durée indéterminée (CDI).

Depuis plusieurs mois, il effectue 48 heures de travail par semaine.
Il constate cependant que les heures supplémentaires ne sont pas
rémunérées conformément à la majoration légale.

Par ailleurs, les équipements de protection individuelle (EPI)
mis à sa disposition par l’employeur semblent insuffisants au regard
des risques professionnels présents sur son lieu de travail.

Ahmed signale ces problèmes à son responsable hiérarchique et demande
l’intervention de l’inspection du travail.

Quelques jours plus tard, l’employeur lui notifie une décision de
licenciement, sans exposer clairement les motifs de cette décision
et sans que, selon Ahmed, une procédure disciplinaire régulière
ait été suivie."""

CONSIGNES_1 = """Consignes de réponse :

- Distinguer les règles du texte initial des dispositions modifiées
  ou abrogées.
- Citer précisément chaque article et le texte juridique auquel il appartient.
- Ne pas déduire automatiquement qu’une infraction ou qu’un licenciement
  abusif est établi : expliquer les conditions et les éléments à vérifier.
- Indiquer la date de validité de chaque disposition utilisée.
- Signaler clairement toute information que les textes disponibles
  ne permettent pas de confirmer.
- Présenter chaque réponse sous la forme :
  réponse courte, explication juridique, références précises,
  version applicable et niveau de confiance.

Répondre en français, de manière structurée et juridiquement rigoureuse.
Ne pas inventer de références légales ni de numéros d'articles."""

CONSIGNES_2 = """Consignes de réponse :

- Se référer exclusivement à la législation du travail algérienne.
- Identifier précisément les articles applicables de la loi n° 90-11
  du 21 avril 1990 relative aux relations de travail.
- Distinguer les dispositions du texte initial de celles qui ont été
  modifiées, complétées ou abrogées.
- Ne pas présenter une règle juridique comme applicable sans vérifier
  sa version et sa date de validité.
- Citer précisément chaque article et le texte juridique auquel
  il appartient.
- Ne pas inventer de dispositions, d'articles ou d'exceptions.
- Signaler clairement toute information que les textes disponibles
  ne permettent pas de confirmer.
- Expliquer les conditions d'application de chaque règle juridique.
- Distinguer les obligations de l'employeur des droits du salarié.
- Indiquer la date de validité de chaque disposition utilisée.

Format obligatoire pour chaque réponse :

1. Réponse courte.
2. Explication juridique.
3. Références juridiques précises (loi et numéro d'article).
4. Version applicable et date de validité.
5. Niveau de confiance (élevé, moyen ou faible), accompagné
   d'une justification.

Répondre en français, de manière structurée, précise et juridiquement
rigoureuse."""

Q_AHMED = [
 "Quelle est la durée légale normale du travail en Algérie ?\nLes 48 heures effectuées par Ahmed constituent-elles automatiquement\nune infraction ?",
 "Dans quelles conditions l’employeur peut-il recourir aux heures\nsupplémentaires ? Quel est le taux minimal de majoration applicable ?",
 "Quelles sont les obligations de l’employeur en matière d’hygiène,\nde sécurité et de mise à disposition des équipements de protection\nindividuelle (EPI) ?",
 "Quelles sont les missions et les pouvoirs de l’inspection du travail\nlorsqu’un salarié signale des infractions à la législation du travail ?",
 "Dans quelles conditions un employeur peut-il licencier un salarié\npour faute grave ? Quelles garanties et procédures doivent être respectées ?",
 "Quelle procédure Ahmed doit-il suivre pour contester son licenciement\net tenter de résoudre ce conflit individuel du travail ?",
]
Q_CONTRAT = [
 "Un contrat de travail est-il valable lorsqu'il n'est pas établi\npar écrit ?",
 "L'employeur peut-il recourir librement au contrat de travail\nà durée déterminée (CDD) en Algérie ?\nDans quels cas la législation autorise-t-elle le recours\nà ce type de contrat ?",
 "Un salarié peut-il bénéficier d'un congé annuel rémunéré\nen Algérie, et comment sa durée est-elle calculée selon\nla législation du travail ?\nPréciser la période de référence et les éventuelles limites\nprévues par les textes.",
 "L'employeur peut-il modifier unilatéralement les clauses\nessentielles du contrat de travail ?\nQuelles sont les limites juridiques de son pouvoir de modification\net quels sont les droits du salarié dans cette situation ?",
]

ITEMS = []
for q in Q_AHMED:
    ITEMS.append((q, f"Cas pratique : Ahmed et les infractions présumées au droit du travail algérien\n\nSituation :\n\n{SITUATION}\n\nQuestion :\n\n{q}\n\n{CONSIGNES_1}"))
for q in Q_CONTRAT:
    ITEMS.append((q, f"Questions juridiques : Validité et modification du contrat de travail et droits du salarié en matière de congé annuel en Algérie.\n\nQuestion :\n\n{q}\n\n{CONSIGNES_2}"))
assert len(ITEMS) == 10

In [4]:
results, prompts = [], {}
for i, (q, p) in enumerate(ITEMS, start=1):
    prompts[f"Q{i}"] = p
    ans = generate(p)
    results.append({"id": i, "question": q, "reponse_llm": ans})
    print(f"Q{i} ✓  {CALLS}")

out = Path("data/processed"); out.mkdir(parents=True, exist_ok=True)
(out / "answers.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

# métadonnées de reproductibilité
con = sqlite3.connect(DB)
mid = con.execute("SELECT model_id FROM cache LIMIT 1").fetchone()
con.close()
meta = {
    "model": MODEL,
    "model_id_returned": mid[0] if mid else None,
    "temperature": DEFAULT_CONFIG["temperature"],
    "seed": DEFAULT_CONFIG["seed"],
    "generated_on": datetime.now(timezone.utc).isoformat(),
    "calls": dict(CALLS),          # real / cached
    "prompts": prompts,
    "prompt_sha256": {k: hashlib.sha256(v.encode()).hexdigest() for k, v in prompts.items()},
}
META.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in meta.items() if k not in ("prompts", "prompt_sha256")}, indent=2))

Q1 ✓  {'real': 1, 'cached': 0}
Q2 ✓  {'real': 2, 'cached': 0}
Q3 ✓  {'real': 3, 'cached': 0}
Q4 ✓  {'real': 4, 'cached': 0}
Q5 ✓  {'real': 5, 'cached': 0}
Q6 ✓  {'real': 6, 'cached': 0}
Q7 ✓  {'real': 7, 'cached': 0}
Q8 ✓  {'real': 8, 'cached': 0}
Q9 ✓  {'real': 9, 'cached': 0}
Q10 ✓  {'real': 10, 'cached': 0}
{
  "model": "openai/gpt-oss-120b",
  "model_id_returned": "openai/gpt-oss-120b",
  "temperature": 0,
  "seed": 42,
  "generated_on": "2026-11-19T06:15:14.307288+00:00",
  "calls": {
    "real": 10,
    "cached": 0
  }
}


In [5]:
Path("docs/week1").mkdir(parents=True, exist_ok=True)

def short(t, n=220):
    t = re.sub(r"\s+", " ", t).strip()
    return (t[:n] + "…") if len(t) > n else t

def cited(t):
    arts = sorted(set(re.findall(r"[Aa]rt(?:icle)?s?\.?\s*\d+(?:[-–]\d+)?(?:\s*bis)?", t)))
    return ", ".join(arts[:6]) or "—"

rows = ["| # | Question | Réponse du LLM (abrégée) | Article cité (à vérifier) | Erreur | Type d'erreur |",
        "|---|---|---|---|---|---|"]
for r in results:
    rows.append(f"| {r['id']} | {short(r['question'], 90).replace('|','/')} | "
                f"{short(r['reponse_llm']).replace('|','/')} | {cited(r['reponse_llm'])} |  |  |")

TABLE = Path("docs/week1/llm_without_rag_table.md")
TABLE.write_text("\n".join(rows), encoding="utf-8")

# archive à télécharger (le disque Colab est temporaire)
import shutil
shutil.make_archive("resultats_experience", "zip", ".", "data/processed")
from google.colab import files
files.download("resultats_experience.zip")
files.download(str(TABLE))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
for r in results:
    print("=" * 80)
    print(f"Q{r['id']} : {r['question']}")
    print("-" * 80)
    print(r["reponse_llm"])

Q1 : Quelle est la durée légale normale du travail en Algérie ?
Les 48 heures effectuées par Ahmed constituent-elles automatiquement
une infraction ?
--------------------------------------------------------------------------------
**1. Durée légale normale du travail en Algérie**  

| Réponse courte | La durée légale normale du travail est de **40 heures par semaine** (8 h/jour sur 5 jours). |
|----------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Explication juridique | Le **Code du travail algérien** (Ordonnance n° 90‑08 du 21 juillet 1990) fixe la durée maximale du travail effectif à 40 h / semaine, sauf dérogation prévue par la loi. Cette règle constitue la « durée normale » à laquelle le salarié doit être rémunéré sans majoration. |
| Références précises | **Article 81** du Code du travail : « La durée du travail effectif

In [7]:
meta = json.load(open("data/processed/experiment_metadata.json", encoding="utf-8"))
info = {k: v for k, v in meta.items() if k not in ("prompts", "prompt_sha256")}
print(json.dumps(info, indent=2, ensure_ascii=False))

{
  "model": "openai/gpt-oss-120b",
  "model_id_returned": "openai/gpt-oss-120b",
  "temperature": 0,
  "seed": 42,
  "generated_on": "2026-11-19T06:15:14.307288+00:00",
  "calls": {
    "real": 10,
    "cached": 0
  }
}
